## Using Vision Language Models with transformers

In [1]:
#!pip install transformers

In [10]:
pip install huggingface_hub

In [13]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)


In [14]:
from transformers import AutoProcessor, AutoModelForCausalLM
import torch
from PIL import Image
import requests
from io import BytesIO

In [22]:
pip install qwen-vl-utils[decord]==0.0.8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 107.4 MB/s eta 0:00:00


In [26]:
!git lfs install
!git clone https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct

Git LFS initialized.
Cloning into 'Qwen2.5-VL-3B-Instruct'...
remote: Enumerating objects: 49, done.
remote: Total 49 (delta 0), reused 0 (delta 0), pack-reused 49 (from 1)
Receiving objects: 100% (49/49), 3.62 MiB | 656.00 KiB/s, done.
Resolving deltas: 100% (20/20), done.
Updating files: 100% (14/14), done.
Filtering content: 100% (2/2), 6.99 GiB | 6.26 MiB/s, done.


## Import Libraries

In [31]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

## Load the pre-trained model on the available device


In [32]:
local_model_path = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    local_model_path,
    torch_dtype="auto",
    device_map="auto"
)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

## Load Preprocessor

In [33]:
processor = AutoProcessor.from_pretrained(
    local_model_path,
    min_pixels=256*28*28,
    max_pixels=512*28*28
    )

## Prepare Input Messages

In [34]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQ1d3KvJnOZXihTw78J6q19cptHDHHllQDbwg&s",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

## Preparation for inference

In [ ]:
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# Process vision inputs
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

inputs = inputs.to("cuda")

## Generation of Output (Inference)

In [35]:
generated_ids = model.generate(**inputs, max_new_tokens=20)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['The image depicts a single head of broccoli placed on a wooden surface. The broccoli is cut in half']


## Use of LLaVA’s Mistral-7B multimodal model to process images and text on GPU

In [6]:
# from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
# import torch

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# processor = LlavaNextProcessor.from_pretrained("llava-hf/llava-v1.6-mistral-7b-hf")
# model = LlavaNextForConditionalGeneration.from_pretrained(
#     "llava-hf/llava-v1.6-mistral-7b-hf",
#     torch_dtype=torch.float16,
#     low_cpu_mem_usage = True
# )

In [28]:
# from PIL import Image
# import requests

# url = "https://raw.githubusercontent.com/haotian-liu/LLaVA/1a91fc274d7c35a9b50b3cb29c4247ae5837ce39/images/llava_v1_5_radar.jpg"
# image = Image.open(requests.get(url, stream=True).raw)
# prompt = "[INST] <image>\nWhat is shown in this image? [/INST]"

# inputs = processor(prompt, image, return_tensors="pt").to(device)
# output = model.generate(**inputs, max_new_tokens=100)

In [ ]:
# print(processor.decode(output[0], skip_special_tokens=True))